# ReadyAI CHiRPE ONNX Summarization + Segmented Inference

This notebook is a variant of `04_readyai_segmented_inference_contract.ipynb`.

The difference is the responsibility boundary for **segment summarization**. In
notebook 04, summarization runs *outside* ReadyAI and ReadyAI receives summaries.
Here, ReadyAI receives **segmented raw text** and runs **Phi-3 ONNX summarization
inside ReadyAI** before classifying each segment.

ReadyAI still does not do raw-transcript segmentation or transcript-level
aggregation at this stage.

## Responsibility Boundary

| Step | Location |
| --- | --- |
| Raw transcript ingestion | Outside ReadyAI |
| Raw transcript segmentation | Outside ReadyAI |
| **Segment summarization (Phi-3 ONNX)** | **Inside ReadyAI** |
| CHiRPE model inference on each segment | Inside ReadyAI |
| Transcript-level voting or aggregation | Outside ReadyAI |

Compared with notebook 04, segment summarization has moved from outside to
inside ReadyAI, and it now runs on the local Phi-3 ONNX Runtime GenAI model.

## Outside ReadyAI: Raw Transcript Input

The payload below represents a raw transcript before ReadyAI. This step is
outside ReadyAI. ReadyAI should not receive this raw transcript directly.

In [1]:
raw_transcript_payload = {
    "participant_id": "demo_001",
    "transcript": [
        {"speaker": "interviewer", "text": "Tell me about whether events have had special meaning for you."},
        {"speaker": "interviewee", "text": "Sometimes I feel like television messages are personally directed at me."},
        {"speaker": "interviewer", "text": "Tell me whether people are talking about you or watching you."},
        {"speaker": "interviewee", "text": "At times I worry strangers are watching me when I am outside, but it comes and goes."},
        {"speaker": "interviewer", "text": "Tell me about any trouble concentrating or focusing."},
        {"speaker": "interviewee", "text": "I have mild trouble focusing at school, but I do not get lost or confused about where I am."},
    ],
}

raw_transcript_payload

{'participant_id': 'demo_001',
 'transcript': [{'speaker': 'interviewer',
   'text': 'Tell me about whether events have had special meaning for you.'},
  {'speaker': 'interviewee',
   'text': 'Sometimes I feel like television messages are personally directed at me.'},
  {'speaker': 'interviewer',
   'text': 'Tell me whether people are talking about you or watching you.'},
  {'speaker': 'interviewee',
   'text': 'At times I worry strangers are watching me when I am outside, but it comes and goes.'},
  {'speaker': 'interviewer',
   'text': 'Tell me about any trouble concentrating or focusing.'},
  {'speaker': 'interviewee',
   'text': 'I have mild trouble focusing at school, but I do not get lost or confused about where I am.'}]}

## Outside ReadyAI: Segment the Raw Transcript (No Summarization)

This cell performs **segmentation only** outside ReadyAI. Unlike notebook 04, it
does not summarize. It uses `SymptomSegmenter` directly so the boundary payload
carries each segment's raw `text`, which ReadyAI will summarize internally.

Unmapped segments are dropped, mirroring `TranscriptPreprocessor`.

In [2]:
from chirpe.data.segmentation import SymptomSegmenter

segmenter = SymptomSegmenter(threshold=0.80)
segments = segmenter.segment_transcript(raw_transcript_payload["transcript"])

mapped_segments = [seg for seg in segments if seg.domain != "unmapped"]
if not mapped_segments:
    raise ValueError("Segmentation produced no mapped segments; check the prompts and threshold.")

participant_id = raw_transcript_payload["participant_id"]
segmented_payload = {
    "participant_id": participant_id,
    "segments": [
        {
            "segment_id": f"{participant_id}_seg_{index + 1:03d}",
            "domain": segment.domain,
            "text": segment.get_text(),
            "start_idx": segment.start_idx,
            "end_idx": segment.end_idx,
        }
        for index, segment in enumerate(mapped_segments)
    ],
}

segmented_payload

{'participant_id': 'demo_001',
 'segments': [{'segment_id': 'demo_001_seg_001',
   'domain': 'P1_Unusual_Thoughts',
   'text': 'Tell me about whether events have had special meaning for you. Sometimes I feel like television messages are personally directed at me.',
   'start_idx': 0,
   'end_idx': 1},
  {'segment_id': 'demo_001_seg_002',
   'domain': 'P2_Suspiciousness',
   'text': 'Tell me whether people are talking about you or watching you. At times I worry strangers are watching me when I am outside, but it comes and goes.',
   'start_idx': 2,
   'end_idx': 3},
  {'segment_id': 'demo_001_seg_003',
   'domain': 'P4_Disoriented',
   'text': 'Tell me about any trouble concentrating or focusing. I have mild trouble focusing at school, but I do not get lost or confused about where I am.',
   'start_idx': 4,
   'end_idx': 5}]}

## ReadyAI Input Contract

The object below is the boundary handoff into ReadyAI: a participant ID plus an
ordered list of segments, each carrying **raw segment text** (not a summary).

In [3]:
import json

print(json.dumps(segmented_payload, indent=2))

{
  "participant_id": "demo_001",
  "segments": [
    {
      "segment_id": "demo_001_seg_001",
      "domain": "P1_Unusual_Thoughts",
      "text": "Tell me about whether events have had special meaning for you. Sometimes I feel like television messages are personally directed at me.",
      "start_idx": 0,
      "end_idx": 1
    },
    {
      "segment_id": "demo_001_seg_002",
      "domain": "P2_Suspiciousness",
      "text": "Tell me whether people are talking about you or watching you. At times I worry strangers are watching me when I am outside, but it comes and goes.",
      "start_idx": 2,
      "end_idx": 3
    },
    {
      "segment_id": "demo_001_seg_003",
      "domain": "P4_Disoriented",
      "text": "Tell me about any trouble concentrating or focusing. I have mild trouble focusing at school, but I do not get lost or confused about where I am.",
      "start_idx": 4,
      "end_idx": 5
    }
  ]
}


## Inside ReadyAI: Load the Phi-3 ONNX Summarizer

This is the start of the ReadyAI-owned step. ReadyAI loads the local Phi-3 ONNX
Runtime GenAI summarizer. The model is loaded once and reused across segments.

Requires the `onnx-llm` extra (`pip install -e ".[onnx-llm]"`). If the model is
not present locally, set `phi3_download=True` or pre-download it with
`scripts/experiments/phi3_onnx_test.py --download`.

In [4]:
from pathlib import Path

from chirpe.data.summarizer import Phi3OnnxSummarizer

# The notebook may run from notebooks/ or the repo root, so try both prefixes.
PHI3_MODEL_DIR_CANDIDATES = [
    Path("outputs/local_onnx_llm/phi3-mini-4k-instruct-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4"),
    Path("../outputs/local_onnx_llm/phi3-mini-4k-instruct-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4"),
]
PHI3_MODEL_DIR = next((p for p in PHI3_MODEL_DIR_CANDIDATES if p.exists()), None)

summarizer = Phi3OnnxSummarizer(
    model_dir=str(PHI3_MODEL_DIR) if PHI3_MODEL_DIR else None,
    max_new_tokens=64,
    download=PHI3_MODEL_DIR is None,
)

print("Loaded Phi-3 ONNX summarizer")

Loaded Phi-3 ONNX summarizer


## Inside ReadyAI: Load CHiRPE Model

ReadyAI loads the CHiRPE classifier. Change `MODEL_PATH` to test a different
checkpoint.

In [5]:
from chirpe.models.classifier import CHRClassifier

MODEL_PATH_CANDIDATES = [
    Path("outputs/string_onnx_checkpoints/bert/best_model"),
    Path("../outputs/string_onnx_checkpoints/bert/best_model"),
]
MODEL_PATH = next((path for path in MODEL_PATH_CANDIDATES if path.exists()), MODEL_PATH_CANDIDATES[0])

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}. Update MODEL_PATH to a trained CHiRPE best_model directory."
    )

classifier = CHRClassifier(model_name=str(MODEL_PATH))
classifier.load(MODEL_PATH)

print(f"Loaded CHiRPE model from: {MODEL_PATH}")

Loaded CHiRPE model from: ../outputs/string_onnx_checkpoints/bert/best_model


## Inside ReadyAI: Summarize (ONNX) then Predict per Segment

ReadyAI summarizes each segment's raw `text` with the Phi-3 ONNX model, then runs
`classifier.predict()` on the generated summaries. It deliberately does not call
`predict_with_segments()` because that performs transcript-level aggregation,
which stays outside ReadyAI.

In [6]:
LABELS = {0: "Healthy", 1: "CHR-P"}


def summarize_and_predict_payload(summarizer, classifier, payload, text_field="text"):
    """Run ONNX summarization + CHiRPE on pre-segmented raw text, per segment only."""
    segments = payload.get("segments", [])
    if not segments:
        raise ValueError("Payload must contain a non-empty 'segments' list.")

    summaries = []
    for index, segment in enumerate(segments):
        text = segment.get(text_field)
        if not text:
            raise ValueError(f"Segment {index} is missing required text field: {text_field!r}")
        summaries.append(summarizer.summarize_segment(text))

    predictions, probabilities = classifier.predict(summaries, return_probs=True)

    segment_outputs = []
    for index, (segment, summary, prediction, probability) in enumerate(
        zip(segments, summaries, predictions, probabilities)
    ):
        prediction = int(prediction)
        healthy_prob = float(probability[0])
        chrp_prob = float(probability[1])

        segment_outputs.append(
            {
                "segment_id": segment.get("segment_id", f"segment_{index + 1}"),
                "domain": segment.get("domain", "unknown"),
                "summary": summary,
                "prediction": prediction,
                "label": LABELS[prediction],
                "confidence": float(max(healthy_prob, chrp_prob)),
                "probabilities": {
                    "Healthy": healthy_prob,
                    "CHR-P": chrp_prob,
                },
            }
        )

    return {
        "participant_id": payload.get("participant_id", "unknown"),
        "segments": segment_outputs,
    }

In [7]:
readyai_output = summarize_and_predict_payload(summarizer, classifier, segmented_payload)
readyai_output

{'participant_id': 'demo_001',
 'segments': [{'segment_id': 'demo_001_seg_001',
   'domain': 'P1_Unusual_Thoughts',
   'summary': 'The patient reports experiencing significant personal resonance with media content, suggesting possible delusional thinking. This symptom may impact daily functioning and warrants further assessment for potential psychotic disorders.',
   'prediction': 1,
   'label': 'CHR-P',
   'confidence': 0.6322895884513855,
   'probabilities': {'Healthy': 0.3677104115486145,
    'CHR-P': 0.6322895884513855}},
  {'segment_id': 'demo_001_seg_002',
   'domain': 'P2_Suspiciousness',
   'summary': 'The interviewee expresses concern about potential surveillance, with intermittent episodes of worry about being watched by strangers. This anxiety may impact their daily functioning and warrants further assessment for risk-related behaviors.',
   'prediction': 1,
   'label': 'CHR-P',
   'confidence': 0.564631462097168,
   'probabilities': {'Healthy': 0.43536850810050964,
    'CHR

## ReadyAI Output Contract

The output remains segmented: one model result per input segment, each including
the ONNX-generated `summary`. The ReadyAI response intentionally does not include
a transcript-level prediction, majority vote, average vote, or participant-level
CHR-P decision.

In [8]:
print(json.dumps(readyai_output, indent=2))

{
  "participant_id": "demo_001",
  "segments": [
    {
      "segment_id": "demo_001_seg_001",
      "domain": "P1_Unusual_Thoughts",
      "summary": "The patient reports experiencing significant personal resonance with media content, suggesting possible delusional thinking. This symptom may impact daily functioning and warrants further assessment for potential psychotic disorders.",
      "prediction": 1,
      "label": "CHR-P",
      "confidence": 0.6322895884513855,
      "probabilities": {
        "Healthy": 0.3677104115486145,
        "CHR-P": 0.6322895884513855
      }
    },
    {
      "segment_id": "demo_001_seg_002",
      "domain": "P2_Suspiciousness",
      "summary": "The interviewee expresses concern about potential surveillance, with intermittent episodes of worry about being watched by strangers. This anxiety may impact their daily functioning and warrants further assessment for risk-related behaviors.",
      "prediction": 1,
      "label": "CHR-P",
      "confidence

## Outside ReadyAI: Aggregate Segment Outputs

This cell performs transcript-level post-processing outside ReadyAI. It consumes
ReadyAI's segmented output and produces a participant-level decision. Do not run
this inside the ReadyAI model-serving step.

In [9]:
def outside_readyai_majority_vote(segment_level_output):
    """Example post-processing that remains outside ReadyAI."""
    votes = [segment["prediction"] for segment in segment_level_output["segments"]]
    chrp_votes = sum(vote == 1 for vote in votes)
    healthy_votes = sum(vote == 0 for vote in votes)
    final_prediction = 1 if chrp_votes > healthy_votes else 0
    return {
        "participant_id": segment_level_output["participant_id"],
        "prediction": final_prediction,
        "label": LABELS[final_prediction],
        "chrp_votes": chrp_votes,
        "healthy_votes": healthy_votes,
    }

aggregated_output = outside_readyai_majority_vote(readyai_output)
aggregated_output

{'participant_id': 'demo_001',
 'prediction': 1,
 'label': 'CHR-P',
 'chrp_votes': 3,
 'healthy_votes': 0}

## Summary

What is inside ReadyAI in this notebook:

- Load the Phi-3 ONNX summarizer and the CHiRPE model.
- Accept an already segmented transcript payload carrying raw segment text.
- Summarize each segment with the Phi-3 ONNX model.
- Run CHiRPE on each segment summary.
- Return per-segment predictions/probabilities (with the generated summary).

What stays outside ReadyAI at this stage:

- Convert the raw transcript into segments.
- Aggregate segment outputs into a transcript-level decision.
- Generate final reports or clinician-facing post-processing.